# CDI Layer 2, Layer 3 and Layer 4

Use **SedAI Docker — Python 3.12**. Set the Layer 2 paths and controls in the next cell.

In [ ]:
from __future__ import annotations

import asyncio
import os
from pathlib import Path
from ML.deep_research.layer2.backend.cli import load_dotenv_key
from ML.deep_research.layer2 import create_run as create_layer2_run
from ML.deep_research.layer2 import run_all as run_layer2
from ML.deep_research.layer2.backend.settings import DOMAIN_PLUGIN_PATH, RUNS_DIR
from ML.deep_research.layer2.backend.fs import load_json

FACT_SHEET_PATH = Path("inputs") / "new_fact_sheet.md"
LAYER2_DOMAIN_PLUGIN = DOMAIN_PLUGIN_PATH
LAYER2_REQUIREMENTS = Path("inputs") / "requirement.md"
LAYER2_REASONING_EFFORT = "max"  # low | medium | high | max
WEB_SEARCH_DEPTH = "medium"  # low | medium | high
WEB_SEARCH_VERBOSITY = "low"  # low | medium | high
PUBLIC_INPUT_CONFIRMED = True

load_dotenv_key()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 2.")

L2_DYNAMIC_RUN = create_layer2_run(
    FACT_SHEET_PATH,
    LAYER2_DOMAIN_PLUGIN,
    LAYER2_REQUIREMENTS,
    RUNS_DIR,
    reasoning_effort=LAYER2_REASONING_EFFORT,
    web_search_context_size=WEB_SEARCH_DEPTH,
    web_search_verbosity=WEB_SEARCH_VERBOSITY,
    public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
)
print("Layer 2: running")
try:
    await asyncio.to_thread(run_layer2, L2_DYNAMIC_RUN)
except Exception:
    print("Layer 2: failed — see run.log")
else:
    print(f"Layer 2: {load_json(L2_DYNAMIC_RUN / 'run.json')['status']}")
print(f"Run: {L2_DYNAMIC_RUN}")
print(f"Log: {L2_DYNAMIC_RUN / 'run.log'}")


## Layer 3 — source discovery only

Select a completed Layer 2 schema-9 run and edit source suggestions. This cell sends public-confirmed inputs to OpenAI, discovers sources and attempts access checks, then saves one JSON file per domain. It does not run deep research or synthesis. Resume uses frozen inputs and settings.


In [1]:
from __future__ import annotations

import asyncio
import os
from pathlib import Path

from ML.deep_research.layer2.backend.cli import load_dotenv_key
from ML.deep_research.layer2.backend.fs import load_json
from ML.deep_research.layer3 import create_run as create_layer3_run, run_all as run_layer3
from ML.deep_research.layer3 import upload_documents, create_research_run
from ML.deep_research.layer3.settings import RUNS_DIR as LAYER3_RUNS_DIR

LAYER3_SOURCE_RUN_PATH = "runs/inputs-new-fact-sheet-93b222cb/L2_20260910_181036_fc5e"  # exact L2 folder; blank uses L2_DYNAMIC_RUN
LAYER3_SOURCE_SUGGESTION_PATH = Path("inputs") / "source_suggestion.md"
LAYER3_RESEARCH_INSTRUCTION_PATH = Path("inputs") / "user_research_instruction.md"
LAYER3_RESUME_RUN_PATH = ""  # exact L3 source run; blank creates a fresh run
LAYER3_PREPARED_RUN_PATH = ""  # existing preparation: create a NEW linked research run
LAYER3_UPLOAD_ONLY = False  # True requires LAYER3_RESUME_RUN_PATH; no new searches
LAYER3_MODEL_REASONING_EFFORT = "max"  # low | medium | high | max
LAYER3_RESEARCH_REASONING_EFFORT = "max"  # low | medium | high | max
WEB_SEARCH_DEPTH = "medium"  # low | medium | high
WEB_SEARCH_VERBOSITY = "low"  # low | medium | high
PUBLIC_INPUT_CONFIRMED = True
LAYER3_RETRY_FAILED = True

load_dotenv_key()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 3.")
if LAYER3_UPLOAD_ONLY and not LAYER3_RESUME_RUN_PATH.strip():
    raise RuntimeError("Upload-only requires an explicit LAYER3_RESUME_RUN_PATH.")
if LAYER3_PREPARED_RUN_PATH.strip() and (LAYER3_RESUME_RUN_PATH.strip() or LAYER3_UPLOAD_ONLY):
    raise RuntimeError("Choose linked research, resume, or upload-only—not multiple actions.")
if LAYER3_RESUME_RUN_PATH.strip():
    L3_SOURCE_RUN = Path(LAYER3_RESUME_RUN_PATH)
elif LAYER3_PREPARED_RUN_PATH.strip():
    L3_SOURCE_RUN = create_research_run(
        Path(LAYER3_PREPARED_RUN_PATH), LAYER3_RUNS_DIR,
        research_instruction=LAYER3_RESEARCH_INSTRUCTION_PATH,
        public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
        research_reasoning_effort=LAYER3_RESEARCH_REASONING_EFFORT,
        web_search_context_size=WEB_SEARCH_DEPTH, web_search_verbosity=WEB_SEARCH_VERBOSITY,
    )
else:
    source = LAYER3_SOURCE_RUN_PATH.strip() or globals().get("L2_DYNAMIC_RUN")
    if not source:
        raise RuntimeError("Set LAYER3_SOURCE_RUN_PATH to the completed Layer 2 run.")
    L3_SOURCE_RUN = create_layer3_run(
        Path(source), LAYER3_RUNS_DIR,
        source_suggestion=LAYER3_SOURCE_SUGGESTION_PATH,
        research_instruction=LAYER3_RESEARCH_INSTRUCTION_PATH,
        public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
        reasoning_effort=LAYER3_MODEL_REASONING_EFFORT,
        research_reasoning_effort=LAYER3_RESEARCH_REASONING_EFFORT,
        web_search_context_size=WEB_SEARCH_DEPTH,
        web_search_verbosity=WEB_SEARCH_VERBOSITY,
    )

print("Layer 3: document upload running" if LAYER3_UPLOAD_ONLY else "Layer 3: running")
try:
    action = upload_documents if LAYER3_UPLOAD_ONLY else run_layer3
    await action(L3_SOURCE_RUN, retry_failed=LAYER3_RETRY_FAILED)
except Exception:
    print("Layer 3: failed — see run.log")
finally:
    record = load_json(L3_SOURCE_RUN / "run.json")
    completed = sum(job.get("status") == "complete" for job in record.get("jobs", {}).values())
    uploads = record.get('document_uploads', {}).get('status', 'not enabled')
    status = record.get('status')
    if status == 'complete' and uploads == 'partial':
        status = 'complete with upload warnings'
    print(f"Status: {status} | Completed: {completed}/{len(record.get('domains', []))}")
    print(f"Uploads: {uploads}")
    research = record.get('research', {})
    researched = sum(job.get('status') == 'complete' for job in research.get('jobs', {}).values())
    print(f"Research: {research.get('status', 'not enabled')} | Completed: {researched}/{len(record.get('domains', []))}")
    for domain, job in research.get('jobs', {}).items():
        if budget := job.get('budget'):
            print(f"{domain}: {job['status']} | calls {budget['used']} used, {budget['remaining']} remaining | {budget['phase']}")
    print(f"Reports: {L3_SOURCE_RUN / 'research'}")
    print(f"Run: {L3_SOURCE_RUN}")
    print(f"Sources: {L3_SOURCE_RUN / 'sources'}")
    print(f"Log: {L3_SOURCE_RUN / 'run.log'}")


Layer 3: running
Status: partial | Completed: 10/10
Uploads: partial
Research: partial | Completed: 9/10
source_finder/000001: complete | calls 27 used, 53 remaining | research
source_finder/000002: complete | calls 61 used, 19 remaining | wrap_up
source_finder/000003: complete | calls 34 used, 46 remaining | research
source_finder/000004: complete | calls 38 used, 42 remaining | research
source_finder/000005: complete | calls 61 used, 19 remaining | wrap_up
source_finder/000006: complete | calls 71 used, 9 remaining | finalization
source_finder/000007: complete | calls 28 used, 52 remaining | research
source_finder/000008: complete | calls 64 used, 16 remaining | wrap_up
source_finder/000009: complete | calls 71 used, 9 remaining | finalization
source_finder/000010: failed | calls 12 used, 68 remaining | research
Reports: /app/runs/inputs-new-fact-sheet-93b222cb/L3_20260912_023825_b4ba/research
Run: /app/runs/inputs-new-fact-sheet-93b222cb/L3_20260912_023825_b4ba
Sources: /app/runs/in

In [ ]:
# Layer 4 — live external-influence research over an existing Layer 3 run
from __future__ import annotations

LAYER4_ENABLED = False

if not LAYER4_ENABLED:
    print("Layer 4 is disabled. Enable explicitly only for a historical schema-8 Layer 3 research run.")
else:
    import asyncio
    import os
    from pathlib import Path

    from IPython.display import JSON, Markdown, clear_output, display

    from ML.deep_research.layer2.backend.cli import load_dotenv_key
    from ML.deep_research.layer2.backend.fs import load_json, read_text
    from ML.deep_research.layer3.usage import summarize_usage
    from ML.deep_research.layer4.cli import run_all as run_layer4
    from ML.deep_research.layer4.create_run import create_run as create_layer4_run
    from ML.deep_research.layer4.settings import (
        DOMAIN_NAMES,
        LAYER3_HARNESS_NAME,
        LAYER3_SCHEMA_VERSION,
        SCHEMA_VERSION as LAYER4_SCHEMA_VERSION,
    )

    LAYER4_SOURCE_RUN_PATH = r"runs\inputs-new-fact-sheet-9563041d\L3_20260902_120424_e0de"
    LAYER4_MODEL_REASONING_EFFORT = "high"  # low | medium | high | max
    LAYER4_WEB_SEARCH_DEPTH = "medium"  # low | medium | high
    LAYER4_WEB_SEARCH_VERBOSITY = "low"  # low | medium | high
    LAYER4_PUBLIC_INPUT_CONFIRMED = True
    LAYER4_RETRY_FAILED = False

    load_dotenv_key()
    L3_RUN = Path(LAYER4_SOURCE_RUN_PATH).resolve()
    if not (L3_RUN / "run.json").is_file():
        raise RuntimeError("Set LAYER4_SOURCE_RUN_PATH to a CDI Layer 3 run folder.")
    source_l3 = load_json(L3_RUN / "run.json")
    if (
        source_l3.get("schema_version") != LAYER3_SCHEMA_VERSION
        or source_l3.get("harness") != LAYER3_HARNESS_NAME
    ):
        raise RuntimeError("Layer 4 requires a schema-8 direct-research Layer 3 run.")
    if not LAYER4_PUBLIC_INPUT_CONFIRMED:
        raise RuntimeError("Layer 4 requires explicit confirmation of public or invented input.")
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 4.")

    L4_RUN = None
    for run_json in sorted(
        L3_RUN.parent.glob("L4_*/run.json"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    ):
        candidate = load_json(run_json)
        source_path = candidate.get("source_l3", {}).get("path", "")
        search_options = candidate.get("web_search", {})
        if (
            candidate.get("schema_version") == LAYER4_SCHEMA_VERSION
            and source_path
            and Path(source_path).resolve() == L3_RUN
            and candidate.get("reasoning_effort") == LAYER4_MODEL_REASONING_EFFORT
            and search_options.get("context_size") == LAYER4_WEB_SEARCH_DEPTH
            and search_options.get("verbosity") == LAYER4_WEB_SEARCH_VERBOSITY
        ):
            L4_RUN = run_json.parent
            break
    if L4_RUN is None:
        L4_RUN = create_layer4_run(
            L3_RUN,
            public_input_confirmed=LAYER4_PUBLIC_INPUT_CONFIRMED,
            reasoning_effort=LAYER4_MODEL_REASONING_EFFORT,
            web_search_context_size=LAYER4_WEB_SEARCH_DEPTH,
            web_search_verbosity=LAYER4_WEB_SEARCH_VERBOSITY,
        )

    l4_before = load_json(L4_RUN / "run.json")
    resume_command = f".\\run.ps1 -ResumeL4 '{L4_RUN}'"
    print(f"Layer 4 run: {L4_RUN}")
    print(f"Resume if interrupted: {resume_command}")

    layer4_task = None
    if l4_before.get("status") != "complete" or LAYER4_RETRY_FAILED:
        layer4_task = asyncio.create_task(
            run_layer4(L4_RUN, retry_failed=LAYER4_RETRY_FAILED)
        )
    while layer4_task and not layer4_task.done():
        await asyncio.sleep(5)
        live = load_json(L4_RUN / "run.json")
        domain_records = live.get("execution", {}).get("domains", {})
        stages = [
            (domain, stage, record)
            for domain, records in domain_records.items()
            for stage, record in records.items()
        ]
        running = [f"{domain}/{stage}" for domain, stage, record in stages if record.get("status") == "running"]
        completed = sum(record.get("status") == "complete" for _, _, record in stages)
        saved_outputs = len(list((L4_RUN / "domains").glob("*/*.md")))
        clear_output(wait=True)
        print(f"Layer 4 run: {L4_RUN}")
        print(f"Resume if interrupted: {resume_command}")
        print(f"Active stage: {running[0] if running else 'none'}")
        print(f"Completed domain stages: {completed}/24")
        print(f"Saved domain outputs: {saved_outputs}/24")
        print("Usage:", summarize_usage(L4_RUN))

    L4_ERROR = ""
    if layer4_task:
        try:
            await layer4_task
        except Exception as error:
            L4_ERROR = f"{type(error).__name__}: {error}"
    clear_output(wait=True)

    l4_record = load_json(L4_RUN / "run.json")
    domain_records = l4_record.get("execution", {}).get("domains", {})
    final_record = l4_record.get("execution", {}).get("final", {})
    saved_outputs = sorted((L4_RUN / "domains").glob("*/*.md"))
    final_answer = L4_RUN / "research" / "final.md"
    print(f"Layer 4 run: {L4_RUN}")
    print(f"Status: {l4_record.get('status', 'unknown')}")
    print(f"Resume if interrupted: {resume_command}")
    if L4_ERROR:
        print(f"Execution error: {L4_ERROR}")
    for domain in DOMAIN_NAMES:
        records = domain_records.get(domain, {})
        statuses = ", ".join(
            f"{stage}={record.get('status', 'missing')}"
            for stage, record in records.items()
        )
        print(f"- {domain}: {statuses or 'missing'}")
    failed = [
        record
        for records in domain_records.values()
        for record in records.values()
        if record.get("status") == "failed"
    ]
    if final_record.get("status") == "failed":
        failed.append(final_record)
    for record in failed:
        print(f"Failed stage: {record.get('stage')}/{record.get('actor')} — {record.get('error')}")
    display(Markdown("### Recorded Layer 4 usage"))
    display(JSON(data=summarize_usage(L4_RUN), expanded=True))
    print(f"Saved domain outputs: {len(saved_outputs)}/24")
    display(Markdown("### External-influence synthesis"))
    display(Markdown(read_text(final_answer) if final_answer.is_file() else "Not produced; resume the Layer 4 run shown above."))
